# Avaliação de Detectores de Objetos para um Vigia Eletrônico

Este notebook reúne os procedimentos empregados na comparação dos modelos YOLOv8n, YOLO11n, YOLO26n e SSD300 em imagens marítimas do conjunto MaSTr1325.

São reproduzidas as seguintes etapas:

1. preparação da amostra de 50 imagens;
2. carregamento dos modelos pré-treinados;
3. avaliação qualitativa de duas cenas;
4. avaliação quantitativa com precisão, recall, F1, acurácia e matrizes de confusão;
5. análise do nível mínimo de confiança; e
6. medição do tempo de processamento.

As imagens do MaSTr1325 não são distribuídas com este notebook. Faça o download pela página oficial do conjunto antes da execução. As anotações manuais devem ser fornecidas no arquivo `vigia_50_final.zip`.

## 1. Preparação do ambiente

No Google Colab, selecione uma GPU em **Ambiente de execução > Alterar tipo de ambiente de execução**. A versão do Ultralytics é fixada para preservar a configuração usada na avaliação.

In [ ]:
%pip install -q ultralytics==8.4.117 opencv-python pandas matplotlib seaborn scikit-learn PyYAML

import platform
import torch
import torchvision
import ultralytics

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Preparação das imagens e das anotações

Envie para o Colab o arquivo ZIP do conjunto MaSTr1325 e o arquivo `vigia_50_final.zip`, que contém as anotações manuais exportadas pelo CVAT. O conjunto original é extraído somente para selecionar novamente as mesmas 50 imagens da câmera esquerda.

In [ ]:
from google.colab import files

print("Selecione o ZIP do MaSTr1325 e o arquivo vigia_50_final.zip")
arquivos_enviados = files.upload()
print("Arquivos enviados:", list(arquivos_enviados))

In [ ]:
from pathlib import Path
import zipfile
import shutil
import cv2
import numpy as np
import pandas as pd

RAIZ_CONTENT = Path("/content")
PASTA_MASTR = RAIZ_CONTENT / "mastr1325"
PASTA_AMOSTRA = RAIZ_CONTENT / "amostra_50_imagens"
PASTA_RESULTADOS = RAIZ_CONTENT / "resultados_vigia"
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

zips_dataset = [
    caminho for caminho in RAIZ_CONTENT.glob("*.zip")
    if "vigia_50_final" not in caminho.name
]

if not zips_dataset:
    raise FileNotFoundError("Envie o arquivo ZIP do conjunto MaSTr1325.")

ZIP_DATASET = zips_dataset[0]
ZIP_FINAL = RAIZ_CONTENT / "vigia_50_final.zip"

if not ZIP_FINAL.exists():
    raise FileNotFoundError("Envie o arquivo vigia_50_final.zip.")

PASTA_MASTR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_DATASET, "r") as arquivo_zip:
    arquivo_zip.extractall(PASTA_MASTR)

imagens_esquerda = sorted([
    caminho for caminho in PASTA_MASTR.rglob("*L.jpg")
    if "__MACOSX" not in caminho.parts
    and not caminho.name.startswith("._")
    and cv2.imread(str(caminho)) is not None
])

if len(imagens_esquerda) < 50:
    raise RuntimeError(
        f"Foram encontradas somente {len(imagens_esquerda)} imagens válidas da câmera esquerda."
    )

indices_selecionados = np.linspace(
    0, len(imagens_esquerda) - 1, 50, dtype=int
)

if PASTA_AMOSTRA.exists():
    shutil.rmtree(PASTA_AMOSTRA)
PASTA_AMOSTRA.mkdir(parents=True)

registros = []
for numero, indice in enumerate(indices_selecionados, start=1):
    origem = imagens_esquerda[int(indice)]
    destino = PASTA_AMOSTRA / origem.name
    shutil.copy2(origem, destino)
    registros.append({
        "Número da amostra": numero,
        "Índice no conjunto": int(indice),
        "Nome do arquivo": origem.name,
    })

manifesto = pd.DataFrame(registros)
manifesto.to_csv(
    PASTA_RESULTADOS / "lista_imagens.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Imagens válidas da câmera esquerda:", len(imagens_esquerda))
print("Quantidade selecionada:", len(manifesto))
display(manifesto.head(10))

## 3. Carregamento dos detectores

Os quatro modelos usam pesos previamente treinados. Não é realizado treinamento adicional com as imagens avaliadas.

In [ ]:
# Definição do dispositivo utilizado pelos modelos
# A GPU será utilizada quando estiver disponível no Google Colab.

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Dispositivo utilizado: {DEVICE}")

In [ ]:
# ------------------------------------------------------------------
# Carregando os 4 detectores pre-treinados no COCO
# ------------------------------------------------------------------
from ultralytics import YOLO
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights

modelos = {}

# ---- Familia YOLO (mesma API para as 3 versoes) ----
for nome, pesos in [("YOLOv8", "yolov8n.pt"), ("YOLO11", "yolo11n.pt"), ("YOLO26", "yolo26n.pt")]:
    try:
        modelos[nome] = YOLO(pesos)
        print(f"[ok] {nome:8s} <- {pesos}")
    except Exception as e:
        print(f"[FALHA] {nome}: {e}")
        print("       -> rode '%pip install -U ultralytics' e reinicie o ambiente")

# ---- SSD300 do torchvision ----
pesos_ssd = SSD300_VGG16_Weights.COCO_V1
ssd = ssd300_vgg16(weights=pesos_ssd).eval().to(DEVICE)
transformacao_ssd = pesos_ssd.transforms()
CLASSES_SSD = pesos_ssd.meta["categories"]   # 91 entradas do COCO (com '__background__' e 'N/A')
print(f"[ok] SSD300  <- VGG16 / COCO ({len([c for c in CLASSES_SSD if c not in ('__background__','N/A')])} classes uteis)")

# ---- Quantos parametros tem cada um? ----
print("\n" + "=" * 70)
print(f"{'MODELO':10s} {'PARAMETROS':>14s}   {'CLASSES':>8s}")
print("=" * 70)
for nome, m in modelos.items():
    n_par = sum(p.numel() for p in m.model.parameters())
    print(f"{nome:10s} {n_par:>14,}   {len(m.names):>8d}")
print(f"{'SSD300':10s} {sum(p.numel() for p in ssd.parameters()):>14,}   {len(CLASSES_SSD):>8d}")
print("=" * 70)

In [ ]:
# Parâmetros utilizados na detecção de objetos
# Os mesmos valores serão aplicados aos modelos comparados.

CONF_MIN = 0.25  # confiança mínima para aceitar uma detecção
IOU_NMS = 0.45   # limite para eliminar caixas sobrepostas
IMGSZ = 640      # dimensão utilizada como entrada nos modelos YOLO

print("Parâmetros definidos:")
print(f"Confiança mínima: {CONF_MIN}")
print(f"IoU do NMS: {IOU_NMS}")
print(f"Dimensão de entrada: {IMGSZ}")

In [ ]:
# ------------------------------------------------------------------
# Interface unificada de deteccao + desenho das bounding boxes
# ------------------------------------------------------------------
import hashlib
import torch

def _cor_da_classe(nome_classe):
    """Cor BGR estavel e distinta para cada nome de classe."""
    h = int(hashlib.md5(nome_classe.encode()).hexdigest()[:6], 16)
    return (60 + (h & 0xFF) % 195, 60 + ((h >> 8) & 0xFF) % 195, 60 + ((h >> 16) & 0xFF) % 195)

def detectar_yolo(modelo, imagem_rgb, conf=CONF_MIN, iou=IOU_NMS):
    """Roda um modelo Ultralytics e devolve a lista normalizada de deteccoes."""
    r = modelo.predict(imagem_rgb[:, :, ::-1], conf=conf, iou=iou, imgsz=IMGSZ,
                       device=DEVICE, verbose=False)[0]
    saida = []
    for caixa in r.boxes:
        x1, y1, x2, y2 = caixa.xyxy[0].tolist()
        saida.append({"caixa": (x1, y1, x2, y2),
                      "classe": r.names[int(caixa.cls[0])],
                      "conf": float(caixa.conf[0])})
    return saida

@torch.no_grad()
def detectar_ssd(imagem_rgb, conf=CONF_MIN, modelo=None, nomes_classes=None):
    """Roda o SSD do torchvision e devolve a lista normalizada de deteccoes."""
    modelo = modelo if modelo is not None else ssd
    nomes_classes = nomes_classes if nomes_classes is not None else CLASSES_SSD
    tensor = torch.from_numpy(imagem_rgb.copy()).permute(2, 0, 1).float() / 255.0
    pred = modelo([tensor.to(DEVICE)])[0]
    saida = []
    for caixa, rotulo, escore in zip(pred["boxes"], pred["labels"], pred["scores"]):
        s = float(escore)
        if s < conf:
            continue
        idx = int(rotulo)
        nome = nomes_classes[idx] if idx < len(nomes_classes) else str(idx)
        if nome in ("__background__", "N/A"):
            continue
        saida.append({"caixa": tuple(caixa.tolist()), "classe": nome, "conf": s})
    return saida

def desenhar_deteccoes(imagem_rgb, deteccoes, espessura=None, escala_texto=None):
    """Desenha caixas + label + confidence sobre uma copia da imagem."""
    img = imagem_rgb.copy()
    h, w = img.shape[:2]
    espessura = espessura or max(2, int(round(w / 700)))
    escala_texto = escala_texto or max(0.5, w / 1600)
    for d in deteccoes:
        x1, y1, x2, y2 = [int(v) for v in d["caixa"]]
        cor = _cor_da_classe(d["classe"])
        cv2.rectangle(img, (x1, y1), (x2, y2), cor, espessura)
        texto = f'{d["classe"]} {d["conf"]*100:.0f}%'
        (tw, th), base = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, escala_texto, espessura)
        y_topo = max(y1 - th - base - 3, 0)
        cv2.rectangle(img, (x1, y_topo), (x1 + tw + 4, y_topo + th + base + 3), cor, -1)
        cv2.putText(img, texto, (x1 + 2, y_topo + th + 1),
                    cv2.FONT_HERSHEY_SIMPLEX, escala_texto, (255, 255, 255), espessura, cv2.LINE_AA)
    return img

def detectar_com_todos(imagem_rgb, conf=CONF_MIN):
    """Roda os 4 detectores na mesma imagem e mede o tempo de cada um."""
    import time
    resultado = {}
    for nome, m in modelos.items():
        t0 = time.time()
        det = detectar_yolo(m, imagem_rgb, conf=conf)
        resultado[nome] = {"det": det, "ms": (time.time() - t0) * 1000}
    t0 = time.time()
    resultado["SSD300"] = {"det": detectar_ssd(imagem_rgb, conf=conf), "ms": (time.time() - t0) * 1000}
    return resultado

print("Funcoes prontas: detectar_yolo, detectar_ssd, desenhar_deteccoes, detectar_com_todos")

## 4. Avaliação qualitativa

Duas imagens da câmera esquerda são utilizadas para observar as detecções, as classes, os níveis de confiança e a localização das caixas delimitadoras.

In [ ]:
# Definição das duas imagens que serão utilizadas na comparação
# Todos os modelos receberão exatamente as mesmas imagens.

import cv2
import matplotlib.pyplot as plt

# Índices escolhidos no mosaico
INDICE_IMAGEM_1 = 0
INDICE_IMAGEM_2 = 112

# Caminhos das imagens selecionadas
CAMINHO_IMAGEM_1 = imagens_esquerda[INDICE_IMAGEM_1]
CAMINHO_IMAGEM_2 = imagens_esquerda[INDICE_IMAGEM_2]

# Função para carregar uma imagem e converter de BGR para RGB
def carregar_imagem_rgb(caminho):
    imagem_bgr = cv2.imread(str(caminho))

    if imagem_bgr is None:
        raise FileNotFoundError(
            f"Não foi possível abrir a imagem: {caminho}"
        )

    return cv2.cvtColor(imagem_bgr, cv2.COLOR_BGR2RGB)

# Carregar as duas imagens
imagem_1 = carregar_imagem_rgb(CAMINHO_IMAGEM_1)
imagem_2 = carregar_imagem_rgb(CAMINHO_IMAGEM_2)

# Mostrar as imagens lado a lado para confirmação
fig, eixos = plt.subplots(1, 2, figsize=(16, 6))

eixos[0].imshow(imagem_1)
eixos[0].set_title(
    f"Imagem 1 — índice {INDICE_IMAGEM_1}\n"
    f"{CAMINHO_IMAGEM_1.name}"
)
eixos[0].axis("off")

eixos[1].imshow(imagem_2)
eixos[1].set_title(
    f"Imagem 2 — índice {INDICE_IMAGEM_2}\n"
    f"{CAMINHO_IMAGEM_2.name}"
)
eixos[1].axis("off")

plt.tight_layout()
plt.show()

print("Imagem 1:", CAMINHO_IMAGEM_1)
print("Imagem 2:", CAMINHO_IMAGEM_2)
print("Dimensão da imagem 1:", imagem_1.shape)
print("Dimensão da imagem 2:", imagem_2.shape)

### Salvando separadamente os resultados de cada detector

Nesta etapa, os resultados de YOLOv8, YOLO11, YOLO26 e SSD300 são salvos em arquivos separados para melhorar a apresentação das figuras no artigo.

In [ ]:
# ------------------------------------------------------------------
# Salvar separadamente o resultado de cada detector nas duas imagens
# ------------------------------------------------------------------
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

# Localizar somente as imagens válidas da câmera esquerda
raiz_dataset = Path("/content/mastr1325")

imagens_esquerda = sorted([
    caminho for caminho in raiz_dataset.rglob("*L.jpg")
    if "__MACOSX" not in caminho.parts
])

# Confirmar as mesmas imagens utilizadas anteriormente
caminhos_selecionados = {
    "imagem_1": imagens_esquerda[0],
    "imagem_2": imagens_esquerda[112]
}

pasta_individuais = Path(
    "/content/resultados_vigia/figuras_individuais"
)
pasta_individuais.mkdir(parents=True, exist_ok=True)

nomes_arquivos = {
    "YOLOv8": "yolov8",
    "YOLO11": "yolo11",
    "YOLO26": "yolo26",
    "SSD300": "ssd300"
}

for nome_imagem, caminho_imagem in caminhos_selecionados.items():

    imagem_bgr = cv2.imread(str(caminho_imagem))

    if imagem_bgr is None:
        raise FileNotFoundError(
            f"Não foi possível abrir: {caminho_imagem}"
        )

    imagem_rgb = cv2.cvtColor(
        imagem_bgr,
        cv2.COLOR_BGR2RGB
    )

    resultados = detectar_com_todos(imagem_rgb)

    for nome_modelo, resultado in resultados.items():

        imagem_anotada = desenhar_deteccoes(
            imagem_rgb,
            resultado["det"]
        )

        nome_saida = (
            f"{nome_imagem}_"
            f"{nomes_arquivos[nome_modelo]}.png"
        )

        caminho_saida = pasta_individuais / nome_saida

        # Salvar somente a imagem anotada, sem margens ou títulos pequenos
        plt.imsave(
            caminho_saida,
            imagem_anotada
        )

        print(
            f"[ok] {nome_modelo}: {caminho_saida}"
        )

print("\nImagens individuais salvas em:")
print(pasta_individuais)

3.3 — Comparação dos detectores na Imagem 1

Nesta etapa, YOLOv8, YOLO11, YOLO26 e SSD são aplicados à imagem de índice 0. São apresentados os objetos identificados, os níveis de confiança e o tempo de processamento.

**O que observar no painel:**

1. Quantas detecções foram apresentadas por cada modelo.
2. Se as caixas estão posicionadas corretamente sobre as embarcações.
3. Qual classe foi atribuída ao objeto detectado.
4. Qual foi o nível de confiança da detecção.
5. Quanto tempo cada modelo levou para processar a imagem.
6. Se ocorreram objetos não detectados ou falsos alarmes.

In [ ]:
# ------------------------------------------------------------------
# Comparação dos quatro detectores na Imagem 1
# Imagem selecionada: índice 0
# ------------------------------------------------------------------

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Pasta para salvar os resultados
SAIDAS = Path("/content/resultados_vigia")
SAIDAS.mkdir(parents=True, exist_ok=True)

# Executar uma vez para preparar os modelos e a GPU
print("Preparando os modelos...")
_ = detectar_com_todos(imagem_1)

# Executar novamente para obter os resultados da comparação
print("Executando a comparação...")
resultado_imagem_1 = detectar_com_todos(imagem_1)

# Mostrar os quatro resultados
fig, eixos = plt.subplots(2, 2, figsize=(20, 12))

for eixo, (nome_modelo, resultado) in zip(
    eixos.ravel(),
    resultado_imagem_1.items()
):
    imagem_anotada = desenhar_deteccoes(
        imagem_1,
        resultado["det"]
    )

    eixo.imshow(imagem_anotada)
    eixo.set_title(
        f'{nome_modelo} — {len(resultado["det"])} detecções '
        f'| {resultado["ms"]:.0f} ms',
        fontsize=14
    )
    eixo.axis("off")

plt.suptitle(
    f"Comparação dos detectores — Imagem 1 — "
    f"{CAMINHO_IMAGEM_1.name}",
    fontsize=17
)

plt.tight_layout()

# Salvar a figura
arquivo_figura_1 = SAIDAS / "comparacao_imagem_1.png"

plt.savefig(
    arquivo_figura_1,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

# Criar a tabela de resultados
linhas_imagem_1 = []

for nome_modelo, resultado in resultado_imagem_1.items():
    deteccoes = resultado["det"]

    if not deteccoes:
        linhas_imagem_1.append({
            "Modelo": nome_modelo,
            "Classe": "Nenhuma detecção",
            "Quantidade": 0,
            "Confiança máxima": "-",
            "Tempo (ms)": round(resultado["ms"], 1)
        })
        continue

    tabela_deteccoes = pd.DataFrame(deteccoes)

    for classe, grupo in tabela_deteccoes.groupby("classe"):
        linhas_imagem_1.append({
            "Modelo": nome_modelo,
            "Classe": classe,
            "Quantidade": len(grupo),
            "Confiança máxima":
                f"{grupo['conf'].max() * 100:.1f}%",
            "Tempo (ms)": round(resultado["ms"], 1)
        })

tabela_imagem_1 = pd.DataFrame(linhas_imagem_1)

print("\nResultados da Imagem 1:")
display(tabela_imagem_1)

# Salvar a tabela
arquivo_tabela_1 = SAIDAS / "resultados_imagem_1.csv"

tabela_imagem_1.to_csv(
    arquivo_tabela_1,
    index=False,
    encoding="utf-8-sig"
)

print(f"Figura salva em: {arquivo_figura_1}")
print(f"Tabela salva em: {arquivo_tabela_1}")

## 3.4 — Comparação dos detectores na Imagem 2

Nesta etapa, YOLOv8, YOLO11, YOLO26 e SSD são aplicados à imagem de índice 112. A segunda cena permite verificar se o comportamento dos modelos permanece semelhante em outra configuração do ambiente marítimo.

**O que observar no painel:**

1. Quantas detecções foram apresentadas por cada modelo.
2. Se as caixas estão posicionadas corretamente sobre as embarcações.
3. Qual classe foi atribuída ao objeto detectado.
4. Qual foi o nível de confiança da detecção.
5. Quanto tempo cada modelo levou para processar a imagem.
6. Se ocorreram objetos não detectados ou falsos alarmes.

In [ ]:
# ------------------------------------------------------------------
# Comparação dos quatro detectores na Imagem 2
# Imagem selecionada: índice 112
# ------------------------------------------------------------------

print("Executando a comparação na Imagem 2...")

resultado_imagem_2 = detectar_com_todos(imagem_2)

# Mostrar os quatro resultados
fig, eixos = plt.subplots(2, 2, figsize=(20, 12))

for eixo, (nome_modelo, resultado) in zip(
    eixos.ravel(),
    resultado_imagem_2.items()
):
    imagem_anotada = desenhar_deteccoes(
        imagem_2,
        resultado["det"]
    )

    eixo.imshow(imagem_anotada)
    eixo.set_title(
        f'{nome_modelo} — {len(resultado["det"])} detecções '
        f'| {resultado["ms"]:.0f} ms',
        fontsize=14
    )
    eixo.axis("off")

plt.suptitle(
    f"Comparação dos detectores — Imagem 2 — "
    f"{CAMINHO_IMAGEM_2.name}",
    fontsize=17
)

plt.tight_layout()

# Salvar a figura
arquivo_figura_2 = SAIDAS / "comparacao_imagem_2.png"

plt.savefig(
    arquivo_figura_2,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

# Criar a tabela
linhas_imagem_2 = []

for nome_modelo, resultado in resultado_imagem_2.items():
    deteccoes = resultado["det"]

    if not deteccoes:
        linhas_imagem_2.append({
            "Modelo": nome_modelo,
            "Classe": "Nenhuma detecção",
            "Quantidade": 0,
            "Confiança máxima": "-",
            "Tempo (ms)": round(resultado["ms"], 1)
        })
        continue

    tabela_deteccoes = pd.DataFrame(deteccoes)

    for classe, grupo in tabela_deteccoes.groupby("classe"):
        linhas_imagem_2.append({
            "Modelo": nome_modelo,
            "Classe": classe,
            "Quantidade": len(grupo),
            "Confiança máxima":
                f"{grupo['conf'].max() * 100:.1f}%",
            "Tempo (ms)": round(resultado["ms"], 1)
        })

tabela_imagem_2 = pd.DataFrame(linhas_imagem_2)

print("\nResultados da Imagem 2:")
display(tabela_imagem_2)

# Salvar a tabela
arquivo_tabela_2 = SAIDAS / "resultados_imagem_2.csv"

tabela_imagem_2.to_csv(
    arquivo_tabela_2,
    index=False,
    encoding="utf-8-sig"
)

print(f"Figura salva em: {arquivo_figura_2}")
print(f"Tabela salva em: {arquivo_tabela_2}")

## 3.5 — Consolidação preliminar dos resultados

Os resultados das duas imagens foram verificados visualmente. As detecções duplicadas foram consideradas falsos positivos e os objetos que não receberam caixas foram considerados não detectados. Essa análise é preliminar, pois ainda não utiliza caixas de referência nem cálculo automático de IoU.

In [ ]:
# ------------------------------------------------------------------
# Consolidação da avaliação visual das duas imagens
# As duas cenas possuem quatro objetos de referência no total:
# três embarcações e uma pessoa.
# ------------------------------------------------------------------

import pandas as pd
from IPython.display import display

resultados_visuais = [
    {
        "Modelo": "YOLOv8",
        "Corretas": 4,
        "Falsos positivos": 0,
        "Não detectados": 0
    },
    {
        "Modelo": "YOLO11",
        "Corretas": 4,
        "Falsos positivos": 0,
        "Não detectados": 0
    },
    {
        "Modelo": "YOLO26",
        "Corretas": 3,
        "Falsos positivos": 2,
        "Não detectados": 1
    },
    {
        "Modelo": "SSD300",
        "Corretas": 2,
        "Falsos positivos": 0,
        "Não detectados": 2
    }
]

tabela_consolidada = pd.DataFrame(resultados_visuais)

# Calcular precisão, revocação e F1
tabela_consolidada["Precisão"] = (
    tabela_consolidada["Corretas"]
    / (
        tabela_consolidada["Corretas"]
        + tabela_consolidada["Falsos positivos"]
    )
)

tabela_consolidada["Revocação"] = (
    tabela_consolidada["Corretas"]
    / (
        tabela_consolidada["Corretas"]
        + tabela_consolidada["Não detectados"]
    )
)

tabela_consolidada["F1"] = (
    2
    * tabela_consolidada["Precisão"]
    * tabela_consolidada["Revocação"]
    / (
        tabela_consolidada["Precisão"]
        + tabela_consolidada["Revocação"]
    )
)

# Criar uma versão formatada para exibição
tabela_exibicao = tabela_consolidada.copy()

for coluna in ["Precisão", "Revocação", "F1"]:
    tabela_exibicao[coluna] = tabela_exibicao[coluna].map(
        lambda valor: f"{valor * 100:.1f}%"
    )

print("Resultados consolidados das duas imagens:")
display(tabela_exibicao)

# Salvar a tabela
arquivo_consolidado = (
    SAIDAS / "avaliacao_visual_duas_imagens.csv"
)

tabela_consolidada.to_csv(
    arquivo_consolidado,
    index=False,
    encoding="utf-8-sig"
)

print(f"Tabela salva em: {arquivo_consolidado}")

## 5. Tempo de processamento

Cada modelo é executado 30 vezes em cada uma das duas imagens qualitativas. O desenho das caixas não faz parte da medição.

In [ ]:
# ------------------------------------------------------------------
# Medição repetida do tempo de processamento
# Cada modelo será executado 30 vezes em cada imagem.
# ------------------------------------------------------------------

import time
import numpy as np
import pandas as pd
import torch
from IPython.display import display

REPETICOES = 30
AQUECIMENTOS = 3

def sincronizar_gpu():
    """Aguardar a conclusão das operações executadas pela GPU."""
    if DEVICE == "cuda":
        torch.cuda.synchronize()

def medir_tempo(funcao, imagem, repeticoes=REPETICOES):
    """
    Medir o tempo de processamento de uma função de detecção.
    O desenho das caixas não é incluído na medição.
    """

    # Execuções de aquecimento
    for _ in range(AQUECIMENTOS):
        funcao(imagem)

    sincronizar_gpu()

    tempos = []

    for _ in range(repeticoes):
        sincronizar_gpu()
        inicio = time.perf_counter()

        funcao(imagem)

        sincronizar_gpu()
        fim = time.perf_counter()

        tempos.append((fim - inicio) * 1000)

    return tempos

# Funções associadas a cada modelo
funcoes_modelos = {}

for nome_modelo, modelo in modelos.items():
    funcoes_modelos[nome_modelo] = (
        lambda imagem, modelo=modelo:
        detectar_yolo(modelo, imagem)
    )

funcoes_modelos["SSD300"] = detectar_ssd

# Imagens utilizadas no teste
imagens_teste = {
    "Imagem 1": imagem_1,
    "Imagem 2": imagem_2
}

registros_tempos = []

print(f"Dispositivo: {DEVICE}")

if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"Repetições por modelo e imagem: {REPETICOES}")
print("Iniciando as medições...\n")

# Medir todos os modelos nas duas imagens
for nome_modelo, funcao in funcoes_modelos.items():
    for nome_imagem, imagem in imagens_teste.items():

        print(f"Medindo {nome_modelo} — {nome_imagem}...")

        tempos = medir_tempo(
            funcao,
            imagem,
            repeticoes=REPETICOES
        )

        for numero_execucao, tempo_ms in enumerate(
            tempos,
            start=1
        ):
            registros_tempos.append({
                "Modelo": nome_modelo,
                "Imagem": nome_imagem,
                "Execução": numero_execucao,
                "Tempo (ms)": tempo_ms
            })

# Tabela com todas as execuções
tempos_detalhados = pd.DataFrame(registros_tempos)

# Resumo por modelo, considerando as duas imagens
resumo_tempos = (
    tempos_detalhados
    .groupby("Modelo")["Tempo (ms)"]
    .agg(
        Média="mean",
        Mediana="median",
        Desvio_padrão="std",
        Mínimo="min",
        Máximo="max"
    )
    .reset_index()
)

# Calcular quadros por segundo a partir do tempo médio
resumo_tempos["FPS aproximado"] = (
    1000 / resumo_tempos["Média"]
)

# Arredondar os valores
colunas_numericas = [
    "Média",
    "Mediana",
    "Desvio_padrão",
    "Mínimo",
    "Máximo",
    "FPS aproximado"
]

resumo_tempos[colunas_numericas] = (
    resumo_tempos[colunas_numericas].round(2)
)

print("\nResumo do tempo de processamento:")
display(resumo_tempos)

# Salvar os resultados
arquivo_tempos_detalhados = (
    SAIDAS / "tempos_detalhados.csv"
)

arquivo_resumo_tempos = (
    SAIDAS / "resumo_tempos.csv"
)

tempos_detalhados.to_csv(
    arquivo_tempos_detalhados,
    index=False,
    encoding="utf-8-sig"
)

resumo_tempos.to_csv(
    arquivo_resumo_tempos,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nMedições salvas em: {arquivo_tempos_detalhados}")
print(f"Resumo salvo em: {arquivo_resumo_tempos}")

## 6. Avaliação quantitativa nas 50 imagens

As anotações manuais são extraídas e comparadas com as detecções usando IoU mínima de 0,50 e limite de confiança de 0,25.

In [ ]:
# ------------------------------------------------------------------
# Extração e conferência das anotações das 50 imagens
# ------------------------------------------------------------------
import zipfile
from pathlib import Path

ZIP_FINAL = Path("/content/vigia_50_final.zip")
PASTA_FINAL = Path("/content/vigia_50_final")
PASTA_LABELS_FINAL = PASTA_FINAL / "labels" / "train"

if not ZIP_FINAL.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado em: {ZIP_FINAL}"
    )

PASTA_FINAL.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_FINAL, "r") as arquivo_zip:
    arquivo_zip.extractall(PASTA_FINAL)

arquivo_train = PASTA_FINAL / "train.txt"

with open(arquivo_train, "r") as arquivo:
    imagens_final = [
        Path(linha.strip()).name
        for linha in arquivo
        if linha.strip()
    ]

arquivos_labels_final = sorted(
    PASTA_LABELS_FINAL.glob("*.txt")
)

imagens_anotadas = {
    arquivo.stem + ".jpg"
    for arquivo in arquivos_labels_final
}

imagens_sem_caixas = [
    nome for nome in imagens_final
    if nome not in imagens_anotadas
]

print("Imagens listadas:", len(imagens_final))
print("Imagens com caixas:", len(arquivos_labels_final))
print("Imagens sem caixas:", len(imagens_sem_caixas))
print("Imagem negativa:", imagens_sem_caixas)

### 4.5 — Avaliação quantitativa nas 50 imagens

YOLOv8, YOLO11, YOLO26 e SSD300 foram aplicados às 50 imagens selecionadas. As detecções das classes *boat* e *person* foram comparadas com as anotações manuais utilizando IoU mínimo de 0,50. A avaliação considera verdadeiros positivos, falsos positivos, objetos não detectados, precisão, recall, F1 e acurácia. Também foram calculados a média e o desvio padrão da acurácia obtida em cada imagem.

In [ ]:
# ------------------------------------------------------------------
# Avaliação quantitativa final nas 50 imagens
# ------------------------------------------------------------------
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

IOU_AVALIACAO = 0.50
CLASSES_AVALIADAS = {"boat", "person"}
NOMES_CLASSES = {0: "boat", 1: "person"}

PASTA_RESULTADOS = Path("/content/resultados_vigia")
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

def iou_final(caixa_a, caixa_b):
    ax1, ay1, ax2, ay2 = caixa_a
    bx1, by1, bx2, by2 = caixa_b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    intersecao = max(0, ix2 - ix1) * max(0, iy2 - iy1)

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    uniao = area_a + area_b - intersecao

    return intersecao / uniao if uniao > 0 else 0


def carregar_referencias_final(nome_imagem, largura, altura):
    arquivo_label = (
        PASTA_LABELS_FINAL /
        f"{Path(nome_imagem).stem}.txt"
    )

    # A imagem negativa não possui arquivo de anotação
    if not arquivo_label.exists():
        return []

    referencias = []

    with open(arquivo_label, "r") as arquivo:
        for linha in arquivo:
            valores = linha.strip().split()

            classe_id = int(valores[0])
            cx, cy, w, h = map(float, valores[1:5])

            referencias.append({
                "classe": NOMES_CLASSES[classe_id],
                "caixa": (
                    (cx - w / 2) * largura,
                    (cy - h / 2) * altura,
                    (cx + w / 2) * largura,
                    (cy + h / 2) * altura
                )
            })

    return referencias


def comparar_deteccoes_final(referencias, deteccoes):
    deteccoes_avaliadas = [
        deteccao for deteccao in deteccoes
        if deteccao["classe"] in CLASSES_AVALIADAS
    ]

    deteccoes_avaliadas = sorted(
        deteccoes_avaliadas,
        key=lambda deteccao: deteccao["conf"],
        reverse=True
    )

    referencias_utilizadas = set()
    tp = 0
    fp = 0

    for deteccao in deteccoes_avaliadas:
        melhor_iou = 0
        melhor_indice = None

        for indice, referencia in enumerate(referencias):
            if indice in referencias_utilizadas:
                continue

            if deteccao["classe"] != referencia["classe"]:
                continue

            valor_iou = iou_final(
                deteccao["caixa"],
                referencia["caixa"]
            )

            if valor_iou > melhor_iou:
                melhor_iou = valor_iou
                melhor_indice = indice

        if (
            melhor_indice is not None and
            melhor_iou >= IOU_AVALIACAO
        ):
            tp += 1
            referencias_utilizadas.add(melhor_indice)
        else:
            fp += 1

    fn = len(referencias) - tp

    return tp, fp, fn, deteccoes_avaliadas


def calcular_metricas_final(tp, fp, fn):
    # Caso especial: imagem negativa corretamente sem detecções
    if tp == 0 and fp == 0 and fn == 0:
        return 1.0, 1.0, 1.0, 1.0

    precisao = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 1

    f1 = (
        2 * precisao * recall / (precisao + recall)
        if (precisao + recall) > 0 else 0
    )

    acuracia = (
        tp / (tp + fp + fn)
        if (tp + fp + fn) > 0 else 1
    )

    return precisao, recall, f1, acuracia


resultados_finais_por_imagem = []
predicoes_finais = {}
referencias_finais = {}

for numero, nome_imagem in enumerate(imagens_final, start=1):
    caminhos = list(Path("/content").rglob(nome_imagem))

    if not caminhos:
        print(f"Imagem não encontrada: {nome_imagem}")
        continue

    imagem_bgr = cv2.imread(str(caminhos[0]))
    imagem_rgb = cv2.cvtColor(
        imagem_bgr,
        cv2.COLOR_BGR2RGB
    )

    altura, largura = imagem_rgb.shape[:2]

    referencias = carregar_referencias_final(
        nome_imagem,
        largura,
        altura
    )

    referencias_finais[nome_imagem] = referencias

    resultados_modelos = detectar_com_todos(
        imagem_rgb,
        conf=CONF_MIN
    )

    for modelo, resultado in resultados_modelos.items():
        tp, fp, fn, deteccoes = comparar_deteccoes_final(
            referencias,
            resultado["det"]
        )

        precisao, recall, f1, acuracia = (
            calcular_metricas_final(tp, fp, fn)
        )

        predicoes_finais[(nome_imagem, modelo)] = deteccoes

        resultados_finais_por_imagem.append({
            "Imagem": nome_imagem,
            "Modelo": modelo,
            "Objetos_reais": len(referencias),
            "Deteccoes": len(deteccoes),
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "Precisao": precisao,
            "Recall": recall,
            "F1": f1,
            "Acuracia": acuracia
        })

    if numero % 10 == 0:
        print(f"{numero} de 50 imagens processadas")


df_final_imagens = pd.DataFrame(
    resultados_finais_por_imagem
)

linhas_resumo_final = []

for modelo, grupo in df_final_imagens.groupby("Modelo"):
    tp = int(grupo["TP"].sum())
    fp = int(grupo["FP"].sum())
    fn = int(grupo["FN"].sum())

    precisao_global = (
        tp / (tp + fp)
        if (tp + fp) > 0 else 0
    )

    recall_global = (
        tp / (tp + fn)
        if (tp + fn) > 0 else 0
    )

    f1_global = (
        2 * precisao_global * recall_global /
        (precisao_global + recall_global)
        if (precisao_global + recall_global) > 0 else 0
    )

    acuracia_global = (
        tp / (tp + fp + fn)
        if (tp + fp + fn) > 0 else 0
    )

    linhas_resumo_final.append({
        "Modelo": modelo,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precisão global (%)": round(precisao_global * 100, 1),
        "Recall global (%)": round(recall_global * 100, 1),
        "F1 global (%)": round(f1_global * 100, 1),
        "Acurácia global (%)": round(acuracia_global * 100, 1),
        "Acurácia média (%)": round(
            grupo["Acuracia"].mean() * 100, 1
        ),
        "Desvio padrão (p.p.)": round(
            grupo["Acuracia"].std() * 100, 1
        )
    })

resumo_final_50 = pd.DataFrame(
    linhas_resumo_final
).sort_values(
    "Recall global (%)",
    ascending=False
).reset_index(drop=True)

print("\nResultados quantitativos das 50 imagens:")
display(resumo_final_50)

df_final_imagens.to_csv(
    PASTA_RESULTADOS /
    "resultados_detalhados_50_imagens.csv",
    index=False
)

resumo_final_50.to_csv(
    PASTA_RESULTADOS /
    "resumo_quantitativo_50_imagens.csv",
    index=False
)

print("\nArquivos salvos em:", PASTA_RESULTADOS)

### 4.6 — Matrizes de confusão

As matrizes de confusão apresentam os acertos e os erros de classificação de cada modelo. As linhas representam as classes anotadas manualmente e as colunas representam as classes estimadas pelos detectores. A classe “Fundo” registra os objetos não detectados e as detecções realizadas sem um objeto correspondente.

In [ ]:
# ------------------------------------------------------------------
# Matrizes de confusão dos quatro modelos
# ------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

CLASSES_MATRIZ = ["boat", "person", "background"]
NOMES_EXIBICAO = ["Embarcação", "Pessoa", "Fundo"]

def construir_matriz_deteccao(referencias, deteccoes):
    matriz = np.zeros((3, 3), dtype=int)

    pares_possiveis = []

    # Procurar correspondências com IoU igual ou superior a 0,50
    for indice_ref, referencia in enumerate(referencias):
        for indice_det, deteccao in enumerate(deteccoes):
            valor_iou = iou_final(
                referencia["caixa"],
                deteccao["caixa"]
            )

            if valor_iou >= IOU_AVALIACAO:
                pares_possiveis.append(
                    (valor_iou, indice_ref, indice_det)
                )

    # Priorizar os pares com maior IoU
    pares_possiveis.sort(reverse=True)

    referencias_usadas = set()
    deteccoes_usadas = set()

    for valor_iou, indice_ref, indice_det in pares_possiveis:
        if indice_ref in referencias_usadas:
            continue

        if indice_det in deteccoes_usadas:
            continue

        classe_real = referencias[indice_ref]["classe"]
        classe_prevista = deteccoes[indice_det]["classe"]

        linha = CLASSES_MATRIZ.index(classe_real)
        coluna = CLASSES_MATRIZ.index(classe_prevista)

        matriz[linha, coluna] += 1

        referencias_usadas.add(indice_ref)
        deteccoes_usadas.add(indice_det)

    # Objetos reais que não foram detectados
    for indice_ref, referencia in enumerate(referencias):
        if indice_ref not in referencias_usadas:
            linha = CLASSES_MATRIZ.index(
                referencia["classe"]
            )
            matriz[linha, 2] += 1

    # Detecções sem objeto correspondente
    for indice_det, deteccao in enumerate(deteccoes):
        if indice_det not in deteccoes_usadas:
            coluna = CLASSES_MATRIZ.index(
                deteccao["classe"]
            )
            matriz[2, coluna] += 1

    return matriz


matrizes_modelos = {}

for modelo in ["YOLOv8", "YOLO11", "YOLO26", "SSD300"]:
    matriz_total = np.zeros((3, 3), dtype=int)

    for nome_imagem in imagens_final:
        referencias = referencias_finais[nome_imagem]
        deteccoes = predicoes_finais[
            (nome_imagem, modelo)
        ]

        matriz_total += construir_matriz_deteccao(
            referencias,
            deteccoes
        )

    matrizes_modelos[modelo] = matriz_total


fig, eixos = plt.subplots(2, 2, figsize=(15, 12))

for eixo, modelo in zip(
    eixos.ravel(),
    ["YOLOv8", "YOLO11", "YOLO26", "SSD300"]
):
    sns.heatmap(
        matrizes_modelos[modelo],
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=NOMES_EXIBICAO,
        yticklabels=NOMES_EXIBICAO,
        ax=eixo
    )

    eixo.set_title(modelo, fontsize=15)
    eixo.set_xlabel("Classe estimada")
    eixo.set_ylabel("Classe real")

plt.suptitle(
    "Matrizes de confusão — 50 imagens",
    fontsize=18
)
plt.tight_layout()

caminho_figura = (
    PASTA_RESULTADOS /
    "matrizes_confusao_50_imagens.png"
)

plt.savefig(
    caminho_figura,
    dpi=200,
    bbox_inches="tight"
)
plt.show()


# Calcular resultados separados por classe
resultados_por_classe = []

for modelo, matriz in matrizes_modelos.items():
    for indice, classe in enumerate(["boat", "person"]):
        tp = matriz[indice, indice]
        fp = matriz[:, indice].sum() - tp
        fn = matriz[indice, :].sum() - tp

        precisao = (
            tp / (tp + fp)
            if (tp + fp) > 0 else 0
        )

        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0 else 0
        )

        f1 = (
            2 * precisao * recall / (precisao + recall)
            if (precisao + recall) > 0 else 0
        )

        resultados_por_classe.append({
            "Modelo": modelo,
            "Classe": (
                "Embarcação"
                if classe == "boat"
                else "Pessoa"
            ),
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "Precisão (%)": round(precisao * 100, 1),
            "Recall (%)": round(recall * 100, 1),
            "F1 (%)": round(f1 * 100, 1)
        })

df_resultados_por_classe = pd.DataFrame(
    resultados_por_classe
)

display(df_resultados_por_classe)

df_resultados_por_classe.to_csv(
    PASTA_RESULTADOS /
    "metricas_por_classe_50_imagens.csv",
    index=False
)

print("Figura salva em:", caminho_figura)

### 4.7 — Análise dos resultados quantitativos

Para o conjunto de 50 imagens, YOLOv8 e YOLO11 apresentaram os maiores valores de recall, ambos com 93,8%. Entretanto, o YOLOv8 produziu menos falsos positivos e alcançou o maior F1 global, apresentando o resultado mais equilibrado entre os modelos avaliados. O YOLO26 detectou todas as pessoas, mas apresentou mais dificuldades na detecção de embarcações. O SSD300 não produziu falsos positivos para a classe embarcação, porém deixou de detectar 20 dos 65 barcos anotados, resultando em recall de 69,2%. Como o sistema “O Vigia” deve evitar que obstáculos deixem de ser identificados, os resultados indicam uma vantagem do YOLOv8 e do YOLO11 para o conjunto avaliado.

## 7. Influência do nível mínimo de confiança

As inferências são obtidas com confiança mínima de 0,01 e posteriormente filtradas entre 0,05 e 0,95, em intervalos de 0,05. Esse procedimento evita repetir a inferência para cada limite avaliado.

In [ ]:
imagens_conf = sorted(PASTA_AMOSTRA.glob("*.jpg"))
DIR_LABELS_CONF = PASTA_LABELS_FINAL
DEVICE_CONF = DEVICE
modelos_conf = modelos
ssd_conf = ssd
classes_ssd_conf = CLASSES_SSD

print("Imagens encontradas:", len(imagens_conf))
print("Pasta das anotações:", DIR_LABELS_CONF)

In [ ]:
CLASSES_AVALIADAS = {0: "boat", 1: "person"}
IOU_MIN_CONF = 0.50
CONFIANCA_INFERENCIA = 0.01
IMGSZ_CONF = 640
IOU_NMS_CONF = 0.45


def carregar_anotacoes_conf(caminho_label, largura, altura):
    anotacoes = []
    caminho_label = Path(caminho_label)

    if not caminho_label.exists():
        return anotacoes

    for linha in caminho_label.read_text().strip().splitlines():
        partes = linha.split()

        if len(partes) < 5:
            continue

        classe_id = int(float(partes[0]))
        cx, cy, largura_caixa, altura_caixa = map(float, partes[1:5])

        x1 = (cx - largura_caixa / 2) * largura
        y1 = (cy - altura_caixa / 2) * altura
        x2 = (cx + largura_caixa / 2) * largura
        y2 = (cy + altura_caixa / 2) * altura

        anotacoes.append({
            "caixa": (x1, y1, x2, y2),
            "classe": CLASSES_AVALIADAS[classe_id]
        })

    return anotacoes


def calcular_iou_conf(caixa_a, caixa_b):
    ax1, ay1, ax2, ay2 = caixa_a
    bx1, by1, bx2, by2 = caixa_b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    intersecao = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)

    uniao = area_a + area_b - intersecao

    return intersecao / uniao if uniao > 0 else 0.0


def inferir_yolo_conf(modelo, imagem_bgr):
    resultado = modelo.predict(
        imagem_bgr,
        conf=CONFIANCA_INFERENCIA,
        iou=IOU_NMS_CONF,
        imgsz=IMGSZ_CONF,
        device=DEVICE_CONF,
        verbose=False
    )[0]

    deteccoes = []

    if resultado.boxes is None:
        return deteccoes

    for caixa in resultado.boxes:
        classe = resultado.names[int(caixa.cls[0])]
        confianca = float(caixa.conf[0])

        if classe not in ("boat", "person"):
            continue

        x1, y1, x2, y2 = caixa.xyxy[0].tolist()

        deteccoes.append({
            "caixa": (x1, y1, x2, y2),
            "classe": classe,
            "conf": confianca
        })

    return deteccoes


@torch.no_grad()
def inferir_ssd_conf(imagem_bgr):
    imagem_rgb = cv2.cvtColor(imagem_bgr, cv2.COLOR_BGR2RGB)

    tensor = (
        torch.from_numpy(imagem_rgb.copy())
        .permute(2, 0, 1)
        .float() / 255.0
    )

    resultado = ssd_conf([tensor.to(DEVICE_CONF)])[0]
    deteccoes = []

    for caixa, rotulo, escore in zip(
        resultado["boxes"],
        resultado["labels"],
        resultado["scores"]
    ):
        confianca = float(escore)

        if confianca < CONFIANCA_INFERENCIA:
            continue

        classe = classes_ssd_conf[int(rotulo)]

        if classe not in ("boat", "person"):
            continue

        deteccoes.append({
            "caixa": tuple(float(v) for v in caixa.tolist()),
            "classe": classe,
            "conf": confianca
        })

    return deteccoes


def avaliar_deteccoes_conf(deteccoes, anotacoes, limite_conf):
    deteccoes_filtradas = [
        deteccao for deteccao in deteccoes
        if deteccao["conf"] >= limite_conf
    ]

    deteccoes_filtradas.sort(
        key=lambda deteccao: deteccao["conf"],
        reverse=True
    )

    anotacoes_utilizadas = set()
    tp = 0
    fp = 0

    for deteccao in deteccoes_filtradas:
        melhor_indice = None
        melhor_iou = 0.0

        for indice, anotacao in enumerate(anotacoes):
            if indice in anotacoes_utilizadas:
                continue

            if deteccao["classe"] != anotacao["classe"]:
                continue

            valor_iou = calcular_iou_conf(
                deteccao["caixa"],
                anotacao["caixa"]
            )

            if valor_iou > melhor_iou:
                melhor_iou = valor_iou
                melhor_indice = indice

        if melhor_indice is not None and melhor_iou >= IOU_MIN_CONF:
            tp += 1
            anotacoes_utilizadas.add(melhor_indice)
        else:
            fp += 1

    fn = len(anotacoes) - len(anotacoes_utilizadas)

    return tp, fp, fn


print("Funções preparadas.")

In [ ]:
import pickle

predicoes_conf = {
    "YOLOv8": {},
    "YOLO11": {},
    "YOLO26": {},
    "SSD300": {}
}

anotacoes_conf = {}

for indice, caminho_imagem in enumerate(imagens_conf, start=1):
    imagem_bgr = cv2.imread(str(caminho_imagem))

    if imagem_bgr is None:
        print("Não foi possível abrir:", caminho_imagem.name)
        continue

    altura, largura = imagem_bgr.shape[:2]

    caminho_label = DIR_LABELS_CONF / f"{caminho_imagem.stem}.txt"

    anotacoes_conf[caminho_imagem.name] = carregar_anotacoes_conf(
        caminho_label,
        largura,
        altura
    )

    for nome_modelo, modelo in modelos_conf.items():
        predicoes_conf[nome_modelo][caminho_imagem.name] = (
            inferir_yolo_conf(modelo, imagem_bgr)
        )

    predicoes_conf["SSD300"][caminho_imagem.name] = (
        inferir_ssd_conf(imagem_bgr)
    )

    if indice % 10 == 0:
        print(f"{indice} de {len(imagens_conf)} imagens processadas")

with open("/content/predicoes_confidence.pkl", "wb") as arquivo:
    pickle.dump(
        {
            "predicoes": predicoes_conf,
            "anotacoes": anotacoes_conf
        },
        arquivo
    )

print("\nInferências concluídas.")
print("Resultados temporários salvos em: /content/predicoes_confidence.pkl")

In [ ]:
limiares_conf = np.round(
    np.arange(0.05, 0.96, 0.05),
    2
)

linhas_conf = []

for nome_modelo in predicoes_conf:
    for limite in limiares_conf:
        tp_total = 0
        fp_total = 0
        fn_total = 0

        for caminho_imagem in imagens_conf:
            nome_imagem = caminho_imagem.name

            tp, fp, fn = avaliar_deteccoes_conf(
                predicoes_conf[nome_modelo][nome_imagem],
                anotacoes_conf[nome_imagem],
                limite
            )

            tp_total += tp
            fp_total += fp
            fn_total += fn

        precisao = (
            tp_total / (tp_total + fp_total)
            if tp_total + fp_total > 0 else 0.0
        )

        recall = (
            tp_total / (tp_total + fn_total)
            if tp_total + fn_total > 0 else 0.0
        )

        f1 = (
            2 * precisao * recall / (precisao + recall)
            if precisao + recall > 0 else 0.0
        )

        linhas_conf.append({
            "Modelo": nome_modelo,
            "Confiança": limite,
            "TP": tp_total,
            "FP": fp_total,
            "FN": fn_total,
            "Precisão (%)": precisao * 100,
            "Recall (%)": recall * 100,
            "F1 (%)": f1 * 100
        })

resultados_conf_df = pd.DataFrame(linhas_conf)

resultados_conf_df.to_csv(
    str(PASTA_RESULTADOS / "resultados_confidence_f1.csv"),
    index=False
)

print("Resultados com confiança igual a 0,25:")
display(
    resultados_conf_df[
        np.isclose(resultados_conf_df["Confiança"], 0.25)
    ].round(1)
)

print("\nArquivo salvo em: /content/resultados_vigia/resultados_confidence_f1.csv")

In [ ]:
cores_conf = {
    "YOLOv8": "#0072B2",
    "YOLO11": "#009E73",
    "YOLO26": "#D55E00",
    "SSD300": "#CC79A7"
}

melhores_limiares = []

plt.figure(figsize=(10, 6))

for nome_modelo in ["YOLOv8", "YOLO11", "YOLO26", "SSD300"]:
    dados_modelo = resultados_conf_df[
        resultados_conf_df["Modelo"] == nome_modelo
    ].sort_values("Confiança")

    indice_melhor = dados_modelo["F1 (%)"].idxmax()
    melhor = dados_modelo.loc[indice_melhor]
    melhores_limiares.append(melhor)

    plt.plot(
        dados_modelo["Confiança"],
        dados_modelo["F1 (%)"],
        marker="o",
        linewidth=2,
        markersize=4,
        color=cores_conf[nome_modelo],
        label=nome_modelo
    )

    plt.scatter(
        melhor["Confiança"],
        melhor["F1 (%)"],
        color=cores_conf[nome_modelo],
        edgecolor="black",
        s=90,
        zorder=5
    )

plt.axvline(
    x=0.25,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="Limite adotado (0,25)"
)

plt.xlabel("Nível mínimo de confiança")
plt.ylabel("F1 score (%)")
plt.title("Influência do nível de confiança sobre o F1 score")
plt.xlim(0.05, 0.95)
plt.ylim(0, 105)
plt.xticks(limiares_conf)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

CAMINHO_GRAFICO_CONF = str(PASTA_RESULTADOS / "grafico_confidence_f1.png")

plt.savefig(
    CAMINHO_GRAFICO_CONF,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

melhores_conf_df = pd.DataFrame(melhores_limiares)[[
    "Modelo",
    "Confiança",
    "TP",
    "FP",
    "FN",
    "Precisão (%)",
    "Recall (%)",
    "F1 (%)"
]].reset_index(drop=True)

print("Melhor nível de confiança encontrado para cada modelo:")
display(melhores_conf_df.round(1))

print("\nGráfico salvo em:", CAMINHO_GRAFICO_CONF)

In [ ]:
print("MELHORES LIMITES EXATOS")
print("-" * 65)

for _, linha in melhores_conf_df.iterrows():
    print(
        f"{linha['Modelo']:8s} | "
        f"confiança = {linha['Confiança']:.2f} | "
        f"Precisão = {linha['Precisão (%)']:.1f}% | "
        f"Recall = {linha['Recall (%)']:.1f}% | "
        f"F1 = {linha['F1 (%)']:.1f}%"
    )

print("\nRESULTADOS COM O LIMITE ADOTADO DE 0,25")
print("-" * 65)

resultado_025 = resultados_conf_df[
    np.isclose(resultados_conf_df["Confiança"], 0.25)
]

for _, linha in resultado_025.iterrows():
    print(
        f"{linha['Modelo']:8s} | "
        f"TP = {int(linha['TP']):2d} | "
        f"FP = {int(linha['FP']):2d} | "
        f"FN = {int(linha['FN']):2d} | "
        f"Precisão = {linha['Precisão (%)']:.1f}% | "
        f"Recall = {linha['Recall (%)']:.1f}% | "
        f"F1 = {linha['F1 (%)']:.1f}%"
    )

In [ ]:
import ultralytics

print("Versão atual do Ultralytics:", ultralytics.__version__)
print("Caminho do YOLO26:", modelos_conf["YOLO26"].ckpt_path)

## 8. Exportação dos resultados

Todos os arquivos gerados são compactados para download ao final da execução.

In [ ]:
# Compactar e baixar todos os resultados finais
import shutil
from google.colab import files

shutil.make_archive(
    "/content/resultados_vigia_50",
    "zip",
    "/content/resultados_vigia"
)

files.download("/content/resultados_vigia_50.zip")

## Considerações sobre a reprodução

Os valores de tempo podem variar conforme a GPU disponibilizada pelo Google Colab. A versão do Ultralytics, os pesos dos modelos, os limites de confiança e IoU e a seleção das imagens devem ser mantidos para permitir a comparação com os resultados registrados no TCC.